In [1]:
import os


os.environ.setdefault("HF_HOME", "/nfsd/lttm4/tesisti/shahrampour/hf")
os.environ.setdefault("HF_DATASETS_CACHE", "/nfsd/lttm4/tesisti/shahrampour/hf_datasets")
os.environ.setdefault("TRANSFORMERS_CACHE", "/nfsd/lttm4/tesisti/shahrampour/hf_transformers")

for k in ["HF_HOME","HF_DATASETS_CACHE","TRANSFORMERS_CACHE"]:
    os.makedirs(os.environ[k], exist_ok=True)

print("HF_HOME:", os.environ["HF_HOME"])
print("HF_DATASETS_CACHE:", os.environ["HF_DATASETS_CACHE"])
print("TRANSFORMERS_CACHE:", os.environ["TRANSFORMERS_CACHE"])

HF_HOME: /nfsd/lttm4/tesisti/shahrampour/hf
HF_DATASETS_CACHE: /nfsd/lttm4/tesisti/shahrampour/hf_datasets
TRANSFORMERS_CACHE: /nfsd/lttm4/tesisti/shahrampour/hf_transformers


## 1) Imports

In [14]:
import numpy as np
import torch
import json
import pandas as pd
import matplotlib.pyplot as plt
import os
import shutil
import copy
import torch.nn.functional as F

from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from safetensors.torch import load_file as safe_load_file
from datasets import load_dataset
from transformers import (
    AutoConfig,
    AutoImageProcessor,
    AutoModelForImageClassification,
    Trainer,
    TrainingArguments,
)
from transformers.trainer_utils import set_seed

from peft import LoraConfig, get_peft_model, TaskType

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

torch: 2.5.1+cu121
cuda available: True
gpu: NVIDIA GeForce GTX 1650 Ti


In [4]:
RUN_NAME = "cifar100_5step_e15_10_5_15_V5.1"

RESULTS_DIR = os.path.join("results", RUN_NAME)
PLOTS_DIR = os.path.join(RESULTS_DIR, "plots")
TABLES_DIR = os.path.join(RESULTS_DIR, "tables")
METRICS_DIR = os.path.join(RESULTS_DIR, "metrics")

STEP1_OUT = f"outputs/{RUN_NAME}/step1_scratch"
STEP1_FINAL_OUT = f"outputs/{RUN_NAME}/step1_scratch_final"
JOINT_OUT = f"outputs/{RUN_NAME}/joint_full"
LORA_BANK_DIR = f"outputs/{RUN_NAME}/lora_bank"
ZIP_MERGE_DIR = f"outputs/{RUN_NAME}/zip_merge"
ZIP_FINAL_DIR = os.path.join(ZIP_MERGE_DIR, "final_model")

os.makedirs(LORA_BANK_DIR, exist_ok=True)
os.makedirs(ZIP_MERGE_DIR, exist_ok=True)
os.makedirs(ZIP_FINAL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(f"outputs/{RUN_NAME}", exist_ok=True)

## 2) Load CIFAR-100 (fine labels)

In [5]:
from datasets import Image

dataset = load_dataset("cifar100")
dataset = dataset.rename_column("fine_label", "label")

dataset = dataset.cast_column("img", Image())

class_names = dataset["train"].features["label"].names
num_classes = len(class_names)

print("num_classes:", num_classes)
print("example keys:", dataset["train"][0].keys())
print("first 10 classes:", class_names[:10])

C:\venvs\vit310\lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\nfsd\lttm4\tesisti\shahrampour\hf\hub\datasets--cifar100. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████████████████████████████████████| 10000/10000 [00:00<00:00, 181459.27 examples/s]


num_classes: 100
example keys: dict_keys(['img', 'label', 'coarse_label'])
first 10 classes: ['apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle', 'bicycle', 'bottle']


In [6]:
def make_custom_eval_dataset(class_ids, remap_labels=True):
    test_ds = filter_dataset_by_classes(dataset["test"], class_ids)

    if remap_labels:
        label2new = {orig: i for i, orig in enumerate(sorted(class_ids))}
        new2orig = {v: k for k, v in label2new.items()}

        def remap(ex):
            ex["label"] = label2new[int(ex["label"])]
            return ex

        test_ds = test_ds.map(remap)
    else:
        label2new = None
        new2orig = None

    test_ds.set_transform(preprocess_val)
    return test_ds, label2new, new2orig

## 3) Define incremental class splits (2/5/10 steps)

In [7]:

num_steps = 5  

assert num_classes % num_steps == 0
classes_per_step = num_classes // num_steps

class_splits = [
    list(range(s * classes_per_step, (s + 1) * classes_per_step))
    for s in range(num_steps)
]

print("classes per step:", classes_per_step)
print("split sizes:", [len(x) for x in class_splits])
print("step0 sample:", class_splits[0][:10], "...", class_splits[0][-3:])

classes per step: 20
split sizes: [20, 20, 20, 20, 20]
step0 sample: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] ... [17, 18, 19]


In [8]:
first_step_classes = class_splits[0]
later_step_classes = []
for s in range(1, num_steps):
    later_step_classes.extend(class_splits[s])
all_classes = list(range(num_classes))

print("first step classes:", len(first_step_classes))
print("later step classes:", len(later_step_classes))
print("all classes:", len(all_classes))

first step classes: 20
later step classes: 80
all classes: 100


## 4) Model + preprocessing

In [9]:
# Requested model
model_checkpoint = "google/vit-base-patch16-224"
image_processor = AutoImageProcessor.from_pretrained(model_checkpoint, use_fast=True)

from torchvision import transforms
from PIL import Image
import numpy as np
import torch

size = image_processor.size
if isinstance(size, dict):
    H = size.get("height", 224)
    W = size.get("width", 224)
else:
    H = W = int(size)

train_transform = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.RandomCrop((H, W), padding=8),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.05, contrast=0.05, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=image_processor.image_mean, std=image_processor.image_std),
])

val_transform = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.ToTensor(),
    transforms.Normalize(mean=image_processor.image_mean, std=image_processor.image_std),
])

def to_pil(x):
    if isinstance(x, Image.Image):
        return x.convert("RGB")

    if isinstance(x, dict):
        if "array" in x:
            x = x["array"]
        elif "bytes" in x:
            import io
            return Image.open(io.BytesIO(x["bytes"])).convert("RGB")

    if isinstance(x, list):
        x = np.array(x, dtype=np.uint8)

    if isinstance(x, np.ndarray):
        arr = x.astype(np.uint8)
        arr = np.squeeze(arr)

        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)

        if arr.ndim == 3 and arr.shape[0] in (1, 3) and arr.shape[-1] not in (1, 3):
            arr = np.transpose(arr, (1, 2, 0))

        if not (arr.ndim == 3 and arr.shape[-1] in (1, 3)):
            raise TypeError(f"Unexpected image array shape after squeeze: {arr.shape}")

        if arr.shape[-1] == 1:
            arr = np.repeat(arr, 3, axis=-1)

        return Image.fromarray(arr).convert("RGB")

    return x

def preprocess_train(ex):
    ex["pixel_values"] = [train_transform(to_pil(img)) for img in ex["img"]]
    ex["labels"] = ex["label"]
    return ex

def preprocess_val(ex):
    ex["pixel_values"] = [val_transform(to_pil(img)) for img in ex["img"]]
    ex["labels"] = ex["label"]
    return ex

def collate_fn(examples):
    pixel_values = torch.stack([e["pixel_values"] for e in examples])
    labels = torch.tensor([int(e["labels"]) for e in examples], dtype=torch.long)
    return {"pixel_values": pixel_values, "labels": labels}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": (preds == labels).mean().item()}

C:\venvs\vit310\lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\nfsd\lttm4\tesisti\shahrampour\hf\hub\models--google--vit-base-patch16-224. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


## 5) Build per-step datasets (step / cumulative / full)

In [10]:
def classes_for_step(step_idx: int) -> list[int]:
    return class_splits[step_idx]

def classes_for_cumulative(step_idx: int) -> list[int]:
    out = []
    for s in range(step_idx + 1):
        out.extend(class_splits[s])
    return out

def filter_by_classes(ds, class_ids):
    class_set = set(class_ids)
    ds = ds.with_format(None)
    return ds.filter(lambda x: int(x["label"]) in class_set)

def make_step_datasets(step_idx: int, split_type: str = "new_only", remap_labels: bool = False):
    """
    split_type:
      - 'new_only'   : only classes of this step
      - 'cumulative' : union of classes up to this step
      - 'full'       : all classes (100)
    """
    if split_type == "full":
        class_ids = list(range(num_classes))
    elif split_type == "cumulative":
        class_ids = classes_for_cumulative(step_idx)
    elif split_type == "new_only":
        class_ids = classes_for_step(step_idx)
    else:
        raise ValueError(f"Unknown split_type: {split_type}")

    train_ds = filter_by_classes(dataset["train"], class_ids)
    test_ds  = filter_by_classes(dataset["test"], class_ids)

    if remap_labels:
        label2new = {orig: i for i, orig in enumerate(sorted(class_ids))}
        new2orig = {v: k for k, v in label2new.items()}

        def remap(ex):
            ex["label"] = label2new[int(ex["label"])]
            return ex

        train_ds = train_ds.map(remap)
        test_ds  = test_ds.map(remap)
    else:
        label2new = {c: c for c in class_ids}
        new2orig = {c: c for c in class_ids}

    train_ds = train_ds.with_transform(preprocess_train)
    test_ds = test_ds.with_transform(preprocess_val)

    print(f"[Dataset] Step {step_idx+1} | split_type={split_type}")
    print(f"[Dataset] Classes: {class_ids[:5]} ... {class_ids[-5:]}")
    print(f"[Dataset] Train size: {len(train_ds)} | Test size: {len(test_ds)}")

    return train_ds, test_ds, label2new, new2orig, class_ids

eval_all = dataset["test"].with_transform(preprocess_val)

print("eval_all size:", len(eval_all))

eval_all size: 10000


In [ ]:
from datasets import concatenate_datasets

def sample_replay_examples(ds, class_ids, per_class=50, seed=42):
    pieces = []
    for c in class_ids:
        sub = ds.filter(lambda x: int(x["label"]) == int(c))
        if len(sub) == 0:
            continue
        take_n = min(per_class, len(sub))
        sub = sub.shuffle(seed=seed + int(c)).select(range(take_n))
        pieces.append(sub)

    if len(pieces) == 0:
        return None

    return concatenate_datasets(pieces)


def make_step_train_with_replay(step_idx, per_class=50, seed=42, train_transform=None):
    current_classes = classes_for_step(step_idx)
    train_new = dataset["train"].filter(lambda x: int(x["label"]) in current_classes)

    old_classes = []
    if step_idx > 0:
        old_classes = classes_for_cumulative(step_idx - 1)

    if len(old_classes) == 0:
        out = train_new.shuffle(seed=seed + step_idx)
        if train_transform is not None:
            out = out.with_transform(train_transform)
        return out, current_classes, old_classes

    replay_ds = sample_replay_examples(
        dataset["train"],
        class_ids=old_classes,
        per_class=per_class,
        seed=seed + step_idx,
    )

    if replay_ds is None:
        out = train_new.shuffle(seed=seed + step_idx)
        if train_transform is not None:
            out = out.with_transform(train_transform)
        return out, current_classes, old_classes

    train_mix = concatenate_datasets([train_new, replay_ds]).shuffle(seed=seed + step_idx)

    if train_transform is not None:
        train_mix = train_mix.with_transform(train_transform)

    return train_mix, current_classes, old_classes

## 6) Training recipes (reasonable settings)

In [11]:
set_seed(42)

# SCRATCH_EPOCHS = 1
# LORA_EPOCHS    = 1
# FT_EPOCHS      = 1
# JOINT_EPOCHS   = 1

# SCRATCH_EPOCHS = 15
# LORA_EPOCHS    = 10
# FT_EPOCHS      = 5
# JOINT_EPOCHS   = 15

SCRATCH_EPOCHS = 15
LORA_EPOCHS    = 6
FT_EPOCHS      = 5
JOINT_EPOCHS   = 15

USE_REPLAY = True
REPLAY_PER_CLASS = 150
REPLAY_SEED = 42

BATCH_SCRATCH = 8
ACCUM_SCRATCH = 2

BATCH_LORA    = 16
ACCUM_LORA    = 1

BATCH_FT      = 8
ACCUM_FT      = 2

LR_SCRATCH = 5e-5
LR_LORA    = 1e-4
LR_FT      = 3e-6
LR_JOINT   = 5e-5

WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
SCHED = "cosine"

USE_FP16 = torch.cuda.is_available()

# --- LoRA-specific knobs ---
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.10
LORA_TARGET_MODULES = ["query", "key", "value"]
LORA_MODULES_TO_SAVE = ["classifier"]
LORA_LABEL_SMOOTHING = 0.05

# best checkpoint selection for LoRA:
# use "all_seen" monitor instead of current_step
LORA_MONITOR_SET = "all_seen"


results = []

In [12]:
run_config = {
    "run_name": RUN_NAME,
    "model_checkpoint": model_checkpoint,
    "init_mode": "scratch",
    "num_classes": num_classes,
    "num_steps": num_steps,
    "classes_per_step": classes_per_step,
    "scratch_epochs": SCRATCH_EPOCHS,
    "lora_epochs": LORA_EPOCHS,
    "ft_epochs": FT_EPOCHS,
    "joint_epochs": JOINT_EPOCHS,
    "lr_scratch": LR_SCRATCH,
    "lr_lora": LR_LORA,
    "lr_ft": LR_FT,
    "lr_joint": LR_JOINT,
    "use_replay": USE_REPLAY,
    "replay_per_class": REPLAY_PER_CLASS,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "lora_target_modules": LORA_TARGET_MODULES,
    "lora_modules_to_save": LORA_MODULES_TO_SAVE,
    "lora_label_smoothing": LORA_LABEL_SMOOTHING,
    "lora_monitor_set": LORA_MONITOR_SET,
}
with open(os.path.join(METRICS_DIR, "run_config.json"), "w") as f:
    json.dump(run_config, f, indent=2)

## 7) Step 1: train full ViT from scratch on step 0 classes

In [13]:
import os, shutil

# --- clean old step1 outputs so stale 20-class checkpoints cannot survive ---
if os.path.exists(STEP1_OUT):
    shutil.rmtree(STEP1_OUT)
if os.path.exists(STEP1_FINAL_OUT):
    shutil.rmtree(STEP1_FINAL_OUT)

step1_idx = 0
train_step1, test_step1, label2new_1, new2orig_1, class_ids_1 = make_step_datasets(
    step1_idx, split_type="new_only", remap_labels=False
)

print("Step1 original classes:", class_ids_1[:5], "...", class_ids_1[-5:])
print("Step1 label range:",
      min(int(train_step1[i]["label"]) for i in range(min(200, len(train_step1)))),
      max(int(train_step1[i]["label"]) for i in range(min(200, len(train_step1)))))

config_step1 = AutoConfig.from_pretrained(
    model_checkpoint,
    num_labels=num_classes,
    id2label={i: str(i) for i in range(num_classes)},
    label2id={str(i): i for i in range(num_classes)},
)

model_step1 = AutoModelForImageClassification.from_config(config_step1)

print("Before training - Step1 model num_labels:", model_step1.config.num_labels)
print("Before training - Step1 classifier out_features:", model_step1.classifier.out_features)
assert model_step1.config.num_labels == num_classes
assert model_step1.classifier.out_features == num_classes

args_step1 = TrainingArguments(
    output_dir=STEP1_OUT,
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    num_train_epochs=SCRATCH_EPOCHS,
    learning_rate=LR_SCRATCH,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=SCHED,
    per_device_train_batch_size=BATCH_SCRATCH,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=ACCUM_SCRATCH,
    fp16=USE_FP16,
    dataloader_num_workers=4,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    disable_tqdm=True,
    max_grad_norm=1.0,
)

trainer_step1 = Trainer(
    model=model_step1,
    args=args_step1,
    train_dataset=train_step1,
    eval_dataset=test_step1,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

from transformers.utils.notebook import NotebookProgressCallback
trainer_step1.remove_callback(NotebookProgressCallback)

trainer_step1.train()

eval_step1 = trainer_step1.evaluate()
print({"eval_step1": eval_step1})

trainer_step1.save_model(STEP1_FINAL_OUT)
image_processor.save_pretrained(STEP1_FINAL_OUT)


reloaded_step1 = AutoModelForImageClassification.from_pretrained(STEP1_FINAL_OUT)
print("Reloaded STEP1 num_labels:", reloaded_step1.config.num_labels)
print("Reloaded STEP1 classifier out_features:", reloaded_step1.classifier.out_features)

assert reloaded_step1.config.num_labels == num_classes
assert reloaded_step1.classifier.out_features == num_classes

Filter: 100%|██████████████████████████████████████████████████████████| 10000/10000 [00:00<00:00, 10450.21 examples/s]


[Dataset] Step 1 | split_type=new_only
[Dataset] Classes: [0, 1, 2, 3, 4] ... [15, 16, 17, 18, 19]
[Dataset] Train size: 10000 | Test size: 2000
Step1 original classes: [0, 1, 2, 3, 4] ... [15, 16, 17, 18, 19]
Step1 label range: 0 19
Before training - Step1 model num_labels: 100
Before training - Step1 classifier out_features: 100


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


RuntimeError: DataLoader worker (pid(s) 26024, 26200, 20384, 2980) exited unexpectedly

In [ ]:
print("Init mode:", run_config["init_mode"])
assert run_config["init_mode"] == "scratch"

In [ ]:
step1_log_df = pd.DataFrame(trainer_step1.state.log_history)
step1_log_df.to_csv(os.path.join(TABLES_DIR, "step1_log_history.csv"), index=False)

step1_metrics = {
    "experiment": "step1_scratch",
    "eval_loss": float(eval_step1.get("eval_loss", np.nan)),
    "eval_accuracy": float(eval_step1.get("eval_accuracy", np.nan)),
}

with open(os.path.join(METRICS_DIR, "step1_metrics.json"), "w") as f:
    json.dump(step1_metrics, f, indent=2)

results.append({
    "experiment": "step1_scratch",
    "method": "full_finetune",
    "step": 1,
    "eval_type": "current_step",
    "eval_accuracy": float(eval_step1.get("eval_accuracy", np.nan)),
    "eval_loss": float(eval_step1.get("eval_loss", np.nan)),
})

step1_log_df.tail()

In [ ]:
if "loss" in step1_log_df.columns:
    train_loss_df = step1_log_df[step1_log_df["loss"].notna()]
    if not train_loss_df.empty:
        plt.figure(figsize=(8,5))
        plt.plot(train_loss_df["step"], train_loss_df["loss"])
        plt.xlabel("Step")
        plt.ylabel("Train Loss")
        plt.title("Step 1 Train Loss")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, "step1_train_loss.png"), dpi=200)
        plt.show()

if "eval_accuracy" in step1_log_df.columns:
    eval_df = step1_log_df[step1_log_df["eval_accuracy"].notna()]
    if not eval_df.empty:
        plt.figure(figsize=(8,5))
        plt.plot(eval_df["epoch"], eval_df["eval_accuracy"], marker="o")
        plt.xlabel("Epoch")
        plt.ylabel("Eval Accuracy")
        plt.title("Step 1 Eval Accuracy")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, "step1_eval_accuracy.png"), dpi=200)
        plt.show()

## 8) Step 2: LoRA only (freeze backbone) on top of Step 1 model

In [ ]:
first_step_classes = classes_for_step(0)

def make_eval_dataset(class_ids, name=None):
    test_ds = filter_by_classes(dataset["test"], class_ids)
    test_ds = test_ds.with_transform(preprocess_val)
    if name is not None:
        print(f"[Eval set] {name}: num_classes={len(class_ids)}, size={len(test_ds)}")
    return test_ds

In [ ]:
lora_results = []

for s in range(2, num_steps + 1):
    stale_train_dir = f"outputs/{RUN_NAME}/step_{s}_lora"
    stale_adapter_dir = os.path.join(LORA_BANK_DIR, f"step_{s}_adapter")

    if os.path.exists(stale_train_dir):
        shutil.rmtree(stale_train_dir)
    if os.path.exists(stale_adapter_dir):
        shutil.rmtree(stale_adapter_dir)

first_step_classes = classes_for_step(0)

def _label_range(ds, n=200):
    vals = [int(ds[i]["label"]) for i in range(min(n, len(ds)))]
    return min(vals), max(vals)

# every LoRA adapter is trained on the SAME fixed backbone
base_model_dir = STEP1_FINAL_OUT

for step_idx in range(1, num_steps):
    if USE_REPLAY:
        train_step, class_ids, old_classes = make_step_train_with_replay(
            step_idx=step_idx,
            per_class=REPLAY_PER_CLASS,
            seed=REPLAY_SEED,
            train_transform=preprocess_train,
        )

        _, test_step, label2new, new2orig, _ = make_step_datasets(
            step_idx,
            split_type="new_only",
            remap_labels=False,
        )

        print(f"\n[LoRA+Replay] Step {step_idx+1}")
        print("Current step classes:", class_ids[:5], "...", class_ids[-5:])
        print("Replay old classes count:", len(old_classes))
        print("Train size with replay:", len(train_step))
    else:
        train_step, test_step, label2new, new2orig, class_ids = make_step_datasets(
            step_idx,
            split_type="new_only",
            remap_labels=False,
        )

        print(f"\n[LoRA separate] Step {step_idx+1}")
        print("Current step classes:", class_ids[:5], "...", class_ids[-5:])

    tr_min, tr_max = _label_range(train_step)
    te_min, te_max = _label_range(test_step)

    print("Train label range:", tr_min, tr_max)
    print("Test label range:", te_min, te_max)
    print("Num labels:", num_classes)

    seen_now = classes_for_cumulative(step_idx)
    later_now = [c for c in seen_now if c not in classes_for_step(0)]

    eval_current = test_step
    eval_first = make_eval_dataset(classes_for_step(0), name=f"step{step_idx+1}_first_step")
    eval_later = make_eval_dataset(later_now, name=f"step{step_idx+1}_later_steps") if len(later_now) > 0 else None
    eval_seen  = make_eval_dataset(seen_now, name=f"step{step_idx+1}_all_seen")

    if LORA_MONITOR_SET == "all_seen":
        eval_monitor = eval_seen
        monitor_name = "all_seen"
    else:
        eval_monitor = eval_current
        monitor_name = "current_step"

    print("Loaded fixed base_model_dir:", base_model_dir)
    base_model = AutoModelForImageClassification.from_pretrained(base_model_dir)

    print("Loaded base_model num_labels:", base_model.config.num_labels)
    print("Loaded base_model classifier out_features:", base_model.classifier.out_features)

    assert base_model.config.num_labels == num_classes, (
        f"Loaded base model num_labels={base_model.config.num_labels}, expected {num_classes}"
    )
    assert base_model.classifier.out_features == num_classes, (
        f"Loaded base classifier out_features={base_model.classifier.out_features}, expected {num_classes}"
    )

    assert tr_min >= 0
    assert tr_max < base_model.classifier.out_features, (
        f"Train labels out of range: max label {tr_max}, classifier out_features {base_model.classifier.out_features}"
    )
    assert te_min >= 0
    assert te_max < base_model.classifier.out_features, (
        f"Test labels out of range: max label {te_max}, classifier out_features {base_model.classifier.out_features}"
    )


    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        target_modules=LORA_TARGET_MODULES,
        modules_to_save=LORA_MODULES_TO_SAVE,
    )

    model_lora = get_peft_model(base_model, lora_config)

    print(f"[LoRA] Wrapped model step {step_idx+1} num_labels:", model_lora.config.num_labels)
    print(f"[LoRA] Wrapped model step {step_idx+1} classifier out_features:", model_lora.classifier.out_features)
    print(f"[LoRA] Best-model monitor for step {step_idx+1}: {monitor_name}")

    assert model_lora.config.num_labels == num_classes, (
        f"Wrapped LoRA model num_labels={model_lora.config.num_labels}, expected {num_classes}"
    )
    assert model_lora.classifier.out_features == num_classes, (
        f"Wrapped LoRA classifier out_features={model_lora.classifier.out_features}, expected {num_classes}"
    )

    model_lora.print_trainable_parameters()

    step_lora_train_dir = f"outputs/{RUN_NAME}/step_{step_idx+1}_lora"

    args_lora = TrainingArguments(
        output_dir=step_lora_train_dir,
        remove_unused_columns=False,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        num_train_epochs=LORA_EPOCHS,
        learning_rate=LR_LORA,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type=SCHED,
        per_device_train_batch_size=BATCH_LORA,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=ACCUM_LORA,
        fp16=USE_FP16,
        dataloader_num_workers=4,
        logging_steps=25,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        label_smoothing_factor=LORA_LABEL_SMOOTHING,
        report_to="none",
        max_grad_norm=1.0,
    )

    trainer_lora = Trainer(
        model=model_lora,
        args=args_lora,
        train_dataset=train_step,
        eval_dataset=eval_monitor,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
    )

    trainer_lora.train()

    eval_current_res = trainer_lora.evaluate(eval_dataset=eval_current)
    eval_first_res   = trainer_lora.evaluate(eval_dataset=eval_first)
    eval_later_res   = trainer_lora.evaluate(eval_dataset=eval_later) if eval_later is not None else {"eval_accuracy": np.nan, "eval_loss": np.nan}
    eval_seen_res    = trainer_lora.evaluate(eval_dataset=eval_seen)

    pd.DataFrame(trainer_lora.state.log_history).to_csv(
        os.path.join(TABLES_DIR, f"step{step_idx+1}_lora_log_history.csv"),
        index=False
    )

    # save adapter only
    step_adapter_dir = os.path.join(LORA_BANK_DIR, f"step_{step_idx+1}_adapter")
    os.makedirs(step_adapter_dir, exist_ok=True)

    print(f"[LoRA] Step {step_idx+1} saving adapter to {step_adapter_dir}")
    trainer_lora.model.save_pretrained(step_adapter_dir)
    image_processor.save_pretrained(step_adapter_dir)

    reloaded_base = AutoModelForImageClassification.from_pretrained(base_model_dir)
    reloaded_adapter = PeftModel.from_pretrained(reloaded_base, step_adapter_dir)

    print(f"[LoRA] Reload check step {step_idx+1} num_labels:", reloaded_adapter.config.num_labels)
    print(f"[LoRA] Reload check step {step_idx+1} classifier out_features:", reloaded_adapter.classifier.out_features)

    assert reloaded_adapter.classifier.out_features == num_classes, (
        f"Reloaded adapter classifier out_features={reloaded_adapter.classifier.out_features}, expected {num_classes}"
    )

    adapter_meta = {
        "step": step_idx + 1,
        "base_model_dir": STEP1_FINAL_OUT,
        "class_ids": class_ids,
        "num_labels": num_classes,
        "strategy": "separate_adapter_with_replay" if USE_REPLAY else "separate_adapter_no_backbone_merge",
        "monitor_set": monitor_name,
        "replay_per_class": REPLAY_PER_CLASS if USE_REPLAY else 0,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
    }
    with open(os.path.join(step_adapter_dir, "adapter_meta.json"), "w") as f:
        json.dump(adapter_meta, f, indent=2)

    print(f"[LoRA] Step {step_idx+1} adapter save finished")

    lora_results.extend([
        {
            "experiment": f"lora_step_{step_idx+1}",
            "method": "lora_replay" if USE_REPLAY else "lora_separate",
            "step": step_idx + 1,
            "eval_type": "current_step",
            "eval_accuracy": float(eval_current_res.get("eval_accuracy", np.nan)),
            "eval_loss": float(eval_current_res.get("eval_loss", np.nan)),
        },
        {
            "experiment": f"lora_step_{step_idx+1}",
            "method": "lora_replay" if USE_REPLAY else "lora_separate",
            "step": step_idx + 1,
            "eval_type": "first_step",
            "eval_accuracy": float(eval_first_res.get("eval_accuracy", np.nan)),
            "eval_loss": float(eval_first_res.get("eval_loss", np.nan)),
        },
        {
            "experiment": f"lora_step_{step_idx+1}",
            "method": "lora_replay" if USE_REPLAY else "lora_separate",
            "step": step_idx + 1,
            "eval_type": "later_steps_seen_so_far",
            "eval_accuracy": float(eval_later_res.get("eval_accuracy", np.nan)),
            "eval_loss": float(eval_later_res.get("eval_loss", np.nan)),
        },
        {
            "experiment": f"lora_step_{step_idx+1}",
            "method": "lora_replay" if USE_REPLAY else "lora_separate",
            "step": step_idx + 1,
            "eval_type": "all_seen",
            "eval_accuracy": float(eval_seen_res.get("eval_accuracy", np.nan)),
            "eval_loss": float(eval_seen_res.get("eval_loss", np.nan)),
        },
    ])

print("\nLoRA replay training finished.")

In [ ]:
# Base model B0
B0_dir = STEP1_FINAL_OUT

# Saved adapter dirs
L1_dir = os.path.join(LORA_BANK_DIR, "step_2_adapter")
L2_dir = os.path.join(LORA_BANK_DIR, "step_3_adapter")
L3_dir = os.path.join(LORA_BANK_DIR, "step_4_adapter")
L4_dir = os.path.join(LORA_BANK_DIR, "step_5_adapter")

print("B0_dir:", B0_dir)
print("L1_dir:", L1_dir)
print("L2_dir:", L2_dir)
print("L3_dir:", L3_dir)
print("L4_dir:", L4_dir)

def extract_mergeable_tensors(adapter_dir):
    base_model = AutoModelForImageClassification.from_pretrained(B0_dir)
    peft_model = PeftModel.from_pretrained(base_model, adapter_dir)

    merge_state = {}
    for k, v in peft_model.state_dict().items():
        if "lora_" in k:
            merge_state[k] = v.detach().cpu().clone()
        elif "classifier.modules_to_save.default.weight" in k:
            merge_state[k] = v.detach().cpu().clone()
        elif "classifier.modules_to_save.default.bias" in k:
            merge_state[k] = v.detach().cpu().clone()

    print(f"[extract_mergeable_tensors] {adapter_dir}")
    print("  total kept tensors:", len(merge_state))
    print("  has classifier weight:", any("classifier.modules_to_save.default.weight" in x for x in merge_state.keys()))
    print("  has classifier bias:", any("classifier.modules_to_save.default.bias" in x for x in merge_state.keys()))

    return merge_state


L1 = extract_mergeable_tensors(L1_dir)
L2 = extract_mergeable_tensors(L2_dir)
L3 = extract_mergeable_tensors(L3_dir)
L4 = extract_mergeable_tensors(L4_dir)

print("num tensors in L1:", len(L1))
print("num tensors in L2:", len(L2))
print("num tensors in L3:", len(L3))
print("num tensors in L4:", len(L4))

In [ ]:
assert set(L1.keys()) == set(L2.keys()) == set(L3.keys()) == set(L4.keys())
print("All adapters have the same LoRA keys.")

## Merging (Average Method)


In [ ]:
L12_avg = {}
for k in L1.keys():
    L12_avg[k] = (L1[k] + L2[k]) / 2.0

print("Created averaged adapter: L12_avg")

In [ ]:
L1234_avg = {}
for k in L1.keys():
    L1234_avg[k] = (L1[k] + L2[k] + L3[k] + L4[k]) / 4.0

print("Created averaged adapter: L1234_avg")

In [ ]:
weights_newer = {"L1": 0.1, "L2": 0.2, "L3": 0.3, "L4": 0.4}
weights_balanced = {"L1": 0.2, "L2": 0.2, "L3": 0.25, "L4": 0.35}
weights_old_friendly = {"L1": 0.35, "L2": 0.30, "L3": 0.20, "L4": 0.15}

L1234_weighted_newer = {}
L1234_weighted_balanced = {}
L1234_weighted_old_friendly = {}

for k in L1.keys():
    L1234_weighted_newer[k] = (
        weights_newer["L1"] * L1[k] +
        weights_newer["L2"] * L2[k] +
        weights_newer["L3"] * L3[k] +
        weights_newer["L4"] * L4[k]
    )

    L1234_weighted_balanced[k] = (
        weights_balanced["L1"] * L1[k] +
        weights_balanced["L2"] * L2[k] +
        weights_balanced["L3"] * L3[k] +
        weights_balanced["L4"] * L4[k]
    )

    L1234_weighted_old_friendly[k] = (
        weights_old_friendly["L1"] * L1[k] +
        weights_old_friendly["L2"] * L2[k] +
        weights_old_friendly["L3"] * L3[k] +
        weights_old_friendly["L4"] * L4[k]
    )

M12 = {}
M34 = {}
for k in L1.keys():
    M12[k] = 0.5 * L1[k] + 0.5 * L2[k]
    M34[k] = 0.5 * L3[k] + 0.5 * L4[k]

M1234_pairwise = {}
for k in L1.keys():
    M1234_pairwise[k] = 0.5 * M12[k] + 0.5 * M34[k]

print("Created avg, weighted, old-friendly, and pairwise merged adapters.")

In [ ]:
from copy import deepcopy

def build_merged_model_from_lora_state(merged_lora_state, reference_adapter_dir):
    base_model = AutoModelForImageClassification.from_pretrained(B0_dir)
    model = PeftModel.from_pretrained(base_model, reference_adapter_dir)

    current_state = model.state_dict()

    missing_in_current = []
    for k in merged_lora_state:
        if k not in current_state:
            missing_in_current.append(k)
            continue
        current_state[k] = merged_lora_state[k].to(dtype=current_state[k].dtype)

    if len(missing_in_current) > 0:
        print("[build_merged_model] missing keys in current_state:", missing_in_current[:10])

    load_info = model.load_state_dict(current_state, strict=False)

    print("[build_merged_model] missing_keys after load:", len(load_info.missing_keys))
    print("[build_merged_model] unexpected_keys after load:", len(load_info.unexpected_keys))
    print("[build_merged_model] classifier out_features:", model.classifier.out_features)

    assert model.classifier.out_features == num_classes, (
        f"Merged model classifier out_features={model.classifier.out_features}, expected {num_classes}"
    )

    return model

In [ ]:
model_L12_avg = build_merged_model_from_lora_state(L12_avg, L1_dir)
print("Built model_L12_avg")

In [ ]:
model_L1234_avg = build_merged_model_from_lora_state(L1234_avg, L1_dir)
print("Built model_L1234_avg")

In [ ]:
model_L1234_weighted_newer = build_merged_model_from_lora_state(L1234_weighted_newer, L1_dir)
model_L1234_weighted_balanced = build_merged_model_from_lora_state(L1234_weighted_balanced, L1_dir)
model_L1234_weighted_old_friendly = build_merged_model_from_lora_state(L1234_weighted_old_friendly, L1_dir)
model_M1234_pairwise = build_merged_model_from_lora_state(M1234_pairwise, L1_dir)

print("Built all merged LoRA models.")

In [ ]:
# Final merged LoRA = L1234_avg

eval_first_merged = make_eval_dataset(classes_for_step(0))
eval_later_merged = make_eval_dataset(list(range(classes_per_step, num_classes)))
eval_all_merged   = make_eval_dataset(classes_for_cumulative(num_steps - 1))

trainer_eval_merged = Trainer(
    model=model_L1234_avg,
    args=TrainingArguments(
        output_dir=f"outputs/{RUN_NAME}/eval_L1234_final",
        remove_unused_columns=False,
        per_device_eval_batch_size=32,
        report_to="none",
    ),
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

merged_final_first = trainer_eval_merged.evaluate(eval_dataset=eval_first_merged)
merged_final_later = trainer_eval_merged.evaluate(eval_dataset=eval_later_merged)
merged_final_seen  = trainer_eval_merged.evaluate(eval_dataset=eval_all_merged)

print("Merged LoRA final - first_step:", merged_final_first)
print("Merged LoRA final - later_steps:", merged_final_later)
print("Merged LoRA final - all_seen:", merged_final_seen)

In [ ]:
trainer_eval_weighted = Trainer(
    model=model_L1234_weighted_newer,
    args=TrainingArguments(
        output_dir=f"outputs/{RUN_NAME}/eval_L1234_weighted_newer_final",
        remove_unused_columns=False,
        per_device_eval_batch_size=32,
        report_to="none",
    ),
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

weighted_newer_final_first = trainer_eval_weighted.evaluate(eval_dataset=eval_first_merged)
weighted_newer_final_later = trainer_eval_weighted.evaluate(eval_dataset=eval_later_merged)
weighted_newer_final_seen  = trainer_eval_weighted.evaluate(eval_dataset=eval_all_merged)

print("Weighted newer LoRA final - first_step:", weighted_newer_final_first)
print("Weighted newer LoRA final - later_steps:", weighted_newer_final_later)
print("Weighted newer LoRA final - all_seen:", weighted_newer_final_seen)

In [ ]:
trainer_eval_weighted_balanced = Trainer(
    model=model_L1234_weighted_balanced,
    args=TrainingArguments(
        output_dir=f"outputs/{RUN_NAME}/eval_L1234_weighted_balanced_final",
        remove_unused_columns=False,
        per_device_eval_batch_size=32,
        report_to="none",
    ),
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

weighted_balanced_final_first = trainer_eval_weighted_balanced.evaluate(eval_dataset=eval_first_merged)
weighted_balanced_final_later = trainer_eval_weighted_balanced.evaluate(eval_dataset=eval_later_merged)
weighted_balanced_final_seen  = trainer_eval_weighted_balanced.evaluate(eval_dataset=eval_all_merged)

print("Weighted balanced LoRA final - first_step:", weighted_balanced_final_first)
print("Weighted balanced LoRA final - later_steps:", weighted_balanced_final_later)
print("Weighted balanced LoRA final - all_seen:", weighted_balanced_final_seen)

In [ ]:
trainer_eval_weighted_old = Trainer(
    model=model_L1234_weighted_old_friendly,
    args=TrainingArguments(
        output_dir=f"outputs/{RUN_NAME}/eval_L1234_weighted_old_friendly_final",
        remove_unused_columns=False,
        per_device_eval_batch_size=32,
        report_to="none",
    ),
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

weighted_old_final_first = trainer_eval_weighted_old.evaluate(eval_dataset=eval_first_merged)
weighted_old_final_later = trainer_eval_weighted_old.evaluate(eval_dataset=eval_later_merged)
weighted_old_final_seen  = trainer_eval_weighted_old.evaluate(eval_dataset=eval_all_merged)

print("Weighted old-friendly LoRA final - first_step:", weighted_old_final_first)
print("Weighted old-friendly LoRA final - later_steps:", weighted_old_final_later)
print("Weighted old-friendly LoRA final - all_seen:", weighted_old_final_seen)

In [ ]:
trainer_eval_pairwise = Trainer(
    model=model_M1234_pairwise,
    args=TrainingArguments(
        output_dir=f"outputs/{RUN_NAME}/eval_M1234_pairwise_final",
        remove_unused_columns=False,
        per_device_eval_batch_size=32,
        report_to="none",
    ),
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

pairwise_final_first = trainer_eval_pairwise.evaluate(eval_dataset=eval_first_merged)
pairwise_final_later = trainer_eval_pairwise.evaluate(eval_dataset=eval_later_merged)
pairwise_final_seen  = trainer_eval_pairwise.evaluate(eval_dataset=eval_all_merged)

print("Pairwise merged LoRA final - first_step:", pairwise_final_first)
print("Pairwise merged LoRA final - later_steps:", pairwise_final_later)
print("Pairwise merged LoRA final - all_seen:", pairwise_final_seen)

In [ ]:
seen_classes_L12 = classes_for_cumulative(2)   # steps 1,2,3
eval_L12 = make_eval_dataset(seen_classes_L12)

print("Num classes in eval_L12:", len(seen_classes_L12))

In [ ]:
trainer_eval_L12 = Trainer(
    model=model_L12_avg,
    args=TrainingArguments(
        output_dir=f"outputs/{RUN_NAME}/eval_L12_avg",
        remove_unused_columns=False,
        per_device_eval_batch_size=32,
        report_to="none",
    ),
    eval_dataset=eval_L12,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

res_L12 = trainer_eval_L12.evaluate()
print("L12_avg on seen classes up to step 3:", res_L12)

In [ ]:
seen_classes_L1234 = classes_for_cumulative(4)   # steps 1..5, all classes
eval_L1234 = make_eval_dataset(seen_classes_L1234)

print("Num classes in eval_L1234:", len(seen_classes_L1234))

In [ ]:
trainer_eval_L1234 = Trainer(
    model=model_L1234_avg,
    args=TrainingArguments(
        output_dir=f"outputs/{RUN_NAME}/eval_L1234_avg",
        remove_unused_columns=False,
        per_device_eval_batch_size=32,
        report_to="none",
    ),
    eval_dataset=eval_L1234,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

res_L1234 = trainer_eval_L1234.evaluate()
print("L1234_avg on all seen classes:", res_L1234)

## 9) Baseline: full fine-tune instead of LoRA (Step 2)

In [ ]:
ft_results = []
base_model_dir = STEP1_FINAL_OUT

for step_idx in range(1, num_steps):
    if USE_REPLAY:
        train_step, class_ids, old_classes = make_step_train_with_replay(
            step_idx=step_idx,
            per_class=REPLAY_PER_CLASS,
            seed=REPLAY_SEED,
            train_transform=preprocess_train,
        )

        _, test_step, label2new, new2orig, _ = make_step_datasets(
            step_idx,
            split_type="new_only",
            remap_labels=False,
        )

        print(f"\n[FT+Replay] Step {step_idx+1}")
        print("Current step classes:", class_ids[:5], "...", class_ids[-5:])
        print("Replay old classes count:", len(old_classes))
        print("Train size with replay:", len(train_step))
    else:
        train_step, test_step, label2new, new2orig, class_ids = make_step_datasets(
            step_idx,
            split_type="new_only",
            remap_labels=False,
        )

        print(f"\n[FT] Step {step_idx+1}")
        print("Current step classes:", class_ids[:5], "...", class_ids[-5:])

    ft_model = AutoModelForImageClassification.from_pretrained(base_model_dir)

    print(f"[FT] Loaded model from {base_model_dir}")
    print(f"[FT] Step {step_idx+1} model num_labels:", ft_model.config.num_labels)
    print(f"[FT] Step {step_idx+1} classifier out_features:", ft_model.classifier.out_features)

    step_ft_dir = f"outputs/{RUN_NAME}/step_{step_idx+1}_ft"

    args_ft = TrainingArguments(
        output_dir=step_ft_dir,
        remove_unused_columns=False,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        num_train_epochs=FT_EPOCHS,
        learning_rate=LR_FT,
        weight_decay=0.01,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type=SCHED,
        per_device_train_batch_size=BATCH_FT,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=ACCUM_FT,
        fp16=USE_FP16,
        dataloader_num_workers=4,
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        report_to="none",
        max_grad_norm=1.0,
    )

    trainer_ft = Trainer(
        model=ft_model,
        args=args_ft,
        train_dataset=train_step,
        eval_dataset=test_step,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
    )

    trainer_ft.train()
    eval_current = trainer_ft.evaluate(test_step)

    seen_classes_now = classes_for_cumulative(step_idx)
    later_classes_now = [c for c in seen_classes_now if c not in classes_for_step(0)]

    ft_test_first = make_eval_dataset(classes_for_step(0))
    ft_test_later = make_eval_dataset(later_classes_now) if len(later_classes_now) > 0 else None
    ft_test_seen  = make_eval_dataset(seen_classes_now)

    ft_eval_first = trainer_ft.evaluate(eval_dataset=ft_test_first)
    ft_eval_later = trainer_ft.evaluate(eval_dataset=ft_test_later) if ft_test_later is not None else {"eval_accuracy": np.nan, "eval_loss": np.nan}
    ft_eval_seen  = trainer_ft.evaluate(eval_dataset=ft_test_seen)

    step_ft_final_dir = f"outputs/{RUN_NAME}/step_{step_idx+1}_ft_final"
    os.makedirs(step_ft_final_dir, exist_ok=True)

    print(f"[FT] Step {step_idx+1} saving model to {step_ft_final_dir}")
    trainer_ft.model.save_pretrained(step_ft_final_dir, safe_serialization=False)
    image_processor.save_pretrained(step_ft_final_dir)
    print(f"[FT] Step {step_idx+1} save finished")

    reloaded_ft = AutoModelForImageClassification.from_pretrained(step_ft_final_dir)
    print(f"[FT] Reload check step {step_idx+1} num_labels:", reloaded_ft.config.num_labels)
    print(f"[FT] Reload check step {step_idx+1} classifier out_features:", reloaded_ft.classifier.out_features)

    pd.DataFrame(trainer_ft.state.log_history).to_csv(
        os.path.join(TABLES_DIR, f"step{step_idx+1}_ft_log_history.csv"),
        index=False
    )

    ft_results.extend([
        {
            "experiment": f"ft_step_{step_idx+1}",
            "method": "ft_replay" if USE_REPLAY else "full_finetune",
            "step": step_idx + 1,
            "eval_type": "current_step",
            "eval_accuracy": float(eval_current.get("eval_accuracy", np.nan)),
            "eval_loss": float(eval_current.get("eval_loss", np.nan)),
        },
        {
            "experiment": f"ft_step_{step_idx+1}",
            "method": "ft_replay" if USE_REPLAY else "full_finetune",
            "step": step_idx + 1,
            "eval_type": "first_step",
            "eval_accuracy": float(ft_eval_first.get("eval_accuracy", np.nan)),
            "eval_loss": float(ft_eval_first.get("eval_loss", np.nan)),
        },
        {
            "experiment": f"ft_step_{step_idx+1}",
            "method": "ft_replay" if USE_REPLAY else "full_finetune",
            "step": step_idx + 1,
            "eval_type": "later_steps_seen_so_far",
            "eval_accuracy": float(ft_eval_later.get("eval_accuracy", np.nan)),
            "eval_loss": float(ft_eval_later.get("eval_loss", np.nan)),
        },
        {
            "experiment": f"ft_step_{step_idx+1}",
            "method": "ft_replay" if USE_REPLAY else "full_finetune",
            "step": step_idx + 1,
            "eval_type": "all_seen",
            "eval_accuracy": float(ft_eval_seen.get("eval_accuracy", np.nan)),
            "eval_loss": float(ft_eval_seen.get("eval_loss", np.nan)),
        },
    ])

    base_model_dir = step_ft_final_dir

print("\nFull FT continual training finished.")

In [ ]:
final_ft_dir = f"outputs/{RUN_NAME}/step_{num_steps}_ft_final"
final_ft_model = AutoModelForImageClassification.from_pretrained(final_ft_dir)

args_ft_eval = TrainingArguments(
    output_dir=f"outputs/{RUN_NAME}/final_ft_eval",
    remove_unused_columns=False,
    report_to="none",
    fp16=USE_FP16,
    per_device_eval_batch_size=32,
)

trainer_ft_final = Trainer(
    model=final_ft_model,
    args=args_ft_eval,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

final_seen_classes = classes_for_cumulative(num_steps - 1)

ft_test_first = make_eval_dataset(first_step_classes)
ft_test_later = make_eval_dataset(later_step_classes)
ft_test_seen = make_eval_dataset(final_seen_classes)

print("Final FT num_labels:", final_ft_model.config.num_labels)
print("Final FT classifier out_features:", final_ft_model.classifier.out_features)
assert final_ft_model.classifier.out_features == num_classes

ft_final_first = trainer_ft_final.evaluate(ft_test_first)
ft_final_later = trainer_ft_final.evaluate(ft_test_later)
ft_final_seen = trainer_ft_final.evaluate(ft_test_seen)

print("FT final - first step:", ft_final_first)
print("FT final - later steps:", ft_final_later)
print("FT final - all seen:", ft_final_seen)

## 10) Upper bound: joint training (full dataset)

In [ ]:
train_joint, test_joint, label2new_J, new2orig_J, class_ids_J = make_step_datasets(
    step_idx=0, split_type="full", remap_labels=False
)

config_joint = AutoConfig.from_pretrained(
    model_checkpoint,
    num_labels=num_classes,
    id2label={i: str(i) for i in range(num_classes)},
    label2id={str(i): i for i in range(num_classes)},
)

joint_model = AutoModelForImageClassification.from_config(config_joint)

args_joint = TrainingArguments(
    output_dir=JOINT_OUT,
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    num_train_epochs=JOINT_EPOCHS,
    learning_rate=LR_JOINT,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=SCHED,
    per_device_train_batch_size=BATCH_FT,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=ACCUM_FT,
    fp16=USE_FP16,
    dataloader_num_workers=4,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    max_grad_norm=1.0,
)

trainer_joint = Trainer(
    model=joint_model,
    args=args_joint,
    train_dataset=train_joint,
    eval_dataset=test_joint,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

trainer_joint.train()

eval_joint = trainer_joint.evaluate()
print({"eval_joint_full": eval_joint})

In [ ]:
joint_log_df = pd.DataFrame(trainer_joint.state.log_history)
joint_log_df.to_csv(os.path.join(TABLES_DIR, "joint_log_history.csv"), index=False)

joint_metrics = {
    "experiment": "joint_full",
    "eval_loss": float(eval_joint.get("eval_loss", np.nan)),
    "eval_accuracy": float(eval_joint.get("eval_accuracy", np.nan)),
}

with open(os.path.join(METRICS_DIR, "joint_metrics.json"), "w") as f:
    json.dump(joint_metrics, f, indent=2)

joint_log_df.tail()

In [ ]:
joint_test_first = make_eval_dataset(first_step_classes)
joint_test_later = make_eval_dataset(later_step_classes)
joint_test_all = make_eval_dataset(all_classes)

joint_final_first = trainer_joint.evaluate(joint_test_first)
joint_final_later = trainer_joint.evaluate(joint_test_later)
joint_final_seen = trainer_joint.evaluate(joint_test_all)

print("Joint final - first step:", joint_final_first)
print("Joint final - later steps:", joint_final_later)
print("Joint final - all seen:", joint_final_seen)

## 11) Compare results (step test vs full test)

In [ ]:
def grab_acc(d):
    return float(d["eval_accuracy"]) if "eval_accuracy" in d else np.nan

def grab_loss(d):
    return float(d["eval_loss"]) if "eval_loss" in d else np.nan

all_results = []

# B0
all_results.extend(results)

# LoRA separate / replay
all_results.extend(lora_results)

# Full FT stepwise
all_results.extend(ft_results)

# Final merged LoRA: simple avg
all_results.extend([
    {
        "experiment": "lora_merged_final_eval",
        "method": "lora_merged",
        "step": num_steps,
        "eval_type": "first_step",
        "eval_accuracy": grab_acc(merged_final_first),
        "eval_loss": grab_loss(merged_final_first),
    },
    {
        "experiment": "lora_merged_final_eval",
        "method": "lora_merged",
        "step": num_steps,
        "eval_type": "later_steps",
        "eval_accuracy": grab_acc(merged_final_later),
        "eval_loss": grab_loss(merged_final_later),
    },
    {
        "experiment": "lora_merged_final_eval",
        "method": "lora_merged",
        "step": num_steps,
        "eval_type": "all_seen",
        "eval_accuracy": grab_acc(merged_final_seen),
        "eval_loss": grab_loss(merged_final_seen),
    },
])

# Final weighted newer merged LoRA
all_results.extend([
    {
        "experiment": "lora_weighted_final_eval",
        "method": "lora_weighted",
        "step": num_steps,
        "eval_type": "first_step",
        "eval_accuracy": grab_acc(weighted_newer_final_first),
        "eval_loss": grab_loss(weighted_newer_final_first),
    },
    {
        "experiment": "lora_weighted_final_eval",
        "method": "lora_weighted",
        "step": num_steps,
        "eval_type": "later_steps",
        "eval_accuracy": grab_acc(weighted_newer_final_later),
        "eval_loss": grab_loss(weighted_newer_final_later),
    },
    {
        "experiment": "lora_weighted_final_eval",
        "method": "lora_weighted",
        "step": num_steps,
        "eval_type": "all_seen",
        "eval_accuracy": grab_acc(weighted_newer_final_seen),
        "eval_loss": grab_loss(weighted_newer_final_seen),
    },
])

# Final weighted balanced merged LoRA
all_results.extend([
    {
        "experiment": "lora_weighted_balanced_final_eval",
        "method": "lora_weighted_balanced",
        "step": num_steps,
        "eval_type": "first_step",
        "eval_accuracy": grab_acc(weighted_balanced_final_first),
        "eval_loss": grab_loss(weighted_balanced_final_first),
    },
    {
        "experiment": "lora_weighted_balanced_final_eval",
        "method": "lora_weighted_balanced",
        "step": num_steps,
        "eval_type": "later_steps",
        "eval_accuracy": grab_acc(weighted_balanced_final_later),
        "eval_loss": grab_loss(weighted_balanced_final_later),
    },
    {
        "experiment": "lora_weighted_balanced_final_eval",
        "method": "lora_weighted_balanced",
        "step": num_steps,
        "eval_type": "all_seen",
        "eval_accuracy": grab_acc(weighted_balanced_final_seen),
        "eval_loss": grab_loss(weighted_balanced_final_seen),
    },
])

# Final old-friendly merged LoRA
all_results.extend([
    {
        "experiment": "lora_weighted_old_final_eval",
        "method": "lora_weighted_old",
        "step": num_steps,
        "eval_type": "first_step",
        "eval_accuracy": grab_acc(weighted_old_final_first),
        "eval_loss": grab_loss(weighted_old_final_first),
    },
    {
        "experiment": "lora_weighted_old_final_eval",
        "method": "lora_weighted_old",
        "step": num_steps,
        "eval_type": "later_steps",
        "eval_accuracy": grab_acc(weighted_old_final_later),
        "eval_loss": grab_loss(weighted_old_final_later),
    },
    {
        "experiment": "lora_weighted_old_final_eval",
        "method": "lora_weighted_old",
        "step": num_steps,
        "eval_type": "all_seen",
        "eval_accuracy": grab_acc(weighted_old_final_seen),
        "eval_loss": grab_loss(weighted_old_final_seen),
    },
])

# Final pairwise merged LoRA
all_results.extend([
    {
        "experiment": "lora_pairwise_final_eval",
        "method": "lora_pairwise",
        "step": num_steps,
        "eval_type": "first_step",
        "eval_accuracy": grab_acc(pairwise_final_first),
        "eval_loss": grab_loss(pairwise_final_first),
    },
    {
        "experiment": "lora_pairwise_final_eval",
        "method": "lora_pairwise",
        "step": num_steps,
        "eval_type": "later_steps",
        "eval_accuracy": grab_acc(pairwise_final_later),
        "eval_loss": grab_loss(pairwise_final_later),
    },
    {
        "experiment": "lora_pairwise_final_eval",
        "method": "lora_pairwise",
        "step": num_steps,
        "eval_type": "all_seen",
        "eval_accuracy": grab_acc(pairwise_final_seen),
        "eval_loss": grab_loss(pairwise_final_seen),
    },
])

# Final FT
all_results.extend([
    {
        "experiment": "ft_final_eval",
        "method": "full_finetune",
        "step": num_steps,
        "eval_type": "first_step",
        "eval_accuracy": grab_acc(ft_final_first),
        "eval_loss": grab_loss(ft_final_first),
    },
    {
        "experiment": "ft_final_eval",
        "method": "full_finetune",
        "step": num_steps,
        "eval_type": "later_steps",
        "eval_accuracy": grab_acc(ft_final_later),
        "eval_loss": grab_loss(ft_final_later),
    },
    {
        "experiment": "ft_final_eval",
        "method": "full_finetune",
        "step": num_steps,
        "eval_type": "all_seen",
        "eval_accuracy": grab_acc(ft_final_seen),
        "eval_loss": grab_loss(ft_final_seen),
    },
])

# Final Joint
all_results.extend([
    {
        "experiment": "joint_final_eval",
        "method": "joint_upper_bound",
        "step": num_steps,
        "eval_type": "first_step",
        "eval_accuracy": grab_acc(joint_final_first),
        "eval_loss": grab_loss(joint_final_first),
    },
    {
        "experiment": "joint_final_eval",
        "method": "joint_upper_bound",
        "step": num_steps,
        "eval_type": "later_steps",
        "eval_accuracy": grab_acc(joint_final_later),
        "eval_loss": grab_loss(joint_final_later),
    },
    {
        "experiment": "joint_final_eval",
        "method": "joint_upper_bound",
        "step": num_steps,
        "eval_type": "all_seen",
        "eval_accuracy": grab_acc(joint_final_seen),
        "eval_loss": grab_loss(joint_final_seen),
    },
])

results_df = pd.DataFrame(all_results)

print("Raw method values before mapping:", sorted(results_df["method"].astype(str).unique()))
print(results_df.tail(20))

In [ ]:
results_df_clean = results_df.copy()

results_df_clean = results_df_clean.rename(columns={
    "eval_accuracy": "accuracy",
    "eval_loss": "loss",
    "eval_type": "eval_set"
})

def map_method(x):
    if x == "step1_scratch":
        return "B0"
    elif x == "lora_separate":
        return "LoRA separate"
    elif x == "lora_replay":
        return "LoRA + Replay"
    elif x == "lora_merged":
        return "Merged LoRA"
    elif x == "lora_weighted":
        return "Weighted Merged LoRA"
    elif x == "lora_weighted_balanced":
        return "Weighted Balanced Merged LoRA"
    elif x == "lora_weighted_old":
        return "Old-Friendly Merged LoRA"
    elif x == "lora_pairwise":
        return "Pairwise Merged LoRA"
    elif x == "full_finetune":
        return "Full FT"
    elif x == "ft_replay":
        return "FT + Replay"
    elif x == "joint_upper_bound":
        return "Joint"
    return x

results_df_clean["method"] = results_df_clean["method"].apply(map_method)

def map_eval_stage(exp_name):
    if "final_eval" in str(exp_name):
        return "final"
    return "step"

results_df_clean["eval_stage"] = results_df_clean["experiment"].apply(map_eval_stage)

def map_adapter_name(row):
    if row["method"] == "B0":
        return "B0"

    if row["method"] in ["LoRA separate", "LoRA + Replay"]:
        step = int(row["step"])
        if step == 2:
            return "L1"
        elif step == 3:
            return "L2"
        elif step == 4:
            return "L3"
        elif step == 5:
            return "L4"

    if row["method"] == "Merged LoRA":
        return "L1234_avg"
    if row["method"] == "Weighted Merged LoRA":
        return "L1234_weighted_newer"
    if row["method"] == "Weighted Balanced Merged LoRA":
        return "L1234_weighted_balanced"
    if row["method"] == "Old-Friendly Merged LoRA":
        return "L1234_weighted_old_friendly"
    if row["method"] == "Pairwise Merged LoRA":
        return "M1234_pairwise"

    return np.nan

results_df_clean["adapter_name"] = results_df_clean.apply(map_adapter_name, axis=1)

results_df_clean = results_df_clean[
    ["method", "adapter_name", "step", "eval_stage", "eval_set", "accuracy", "loss"]
].copy()

results_df_clean.to_csv(os.path.join(TABLES_DIR, "results_summary_clean.csv"), index=False)

print("Clean columns:", list(results_df_clean.columns))
print(results_df_clean)

In [ ]:
summary_lines = [
    "Experiment summary",
    "==================",
    "",
]

for _, row in results_df_clean.iterrows():
    acc = row["accuracy"]
    loss = row["loss"]

    acc_str = f"{acc:.4f}" if pd.notna(acc) else "nan"
    loss_str = f"{loss:.4f}" if pd.notna(loss) else "nan"

    summary_lines.append(
        f"method={row['method']} | adapter_name={row['adapter_name']} | "
        f"step={row['step']} | eval_stage={row['eval_stage']} | "
        f"eval_set={row['eval_set']} | accuracy={acc_str} | loss={loss_str}"
    )

with open(os.path.join(RESULTS_DIR, "summary.txt"), "w") as f:
    f.write("\n".join(summary_lines))

print("\n".join(summary_lines))

print("\nColumns in results_df_clean:")
print(list(results_df_clean.columns))

ft_rows = results_df_clean[
    results_df_clean["method"].astype(str).str.contains("FT|Replay", na=False)
].copy()

show_cols = [c for c in ["method", "step", "eval_stage", "eval_set", "accuracy", "loss"] if c in ft_rows.columns]

if len(ft_rows) > 0 and len(show_cols) > 0:
    print("\nFT-related rows:")
    print(
        ft_rows[show_cols].sort_values(
            [c for c in ["method", "step", "eval_stage", "eval_set"] if c in ft_rows.columns]
        )
    )
else:
    print("\nNo FT-related rows found.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

df_final = results_df_clean[
    (results_df_clean["eval_stage"] == "final") &
    (results_df_clean["step"] == num_steps)
].copy()

available_methods = set(df_final["method"].astype(str).unique())

if "FT + Replay" in available_methods:
    ft_name = "FT + Replay"
elif "Full FT" in available_methods:
    ft_name = "Full FT"
else:
    ft_name = None

plot_methods = [
    "Merged LoRA",
    "Weighted Merged LoRA",
    "Weighted Balanced Merged LoRA",
    "Old-Friendly Merged LoRA",
    "Pairwise Merged LoRA",
    ft_name,
    "Joint"
]
plot_methods = [m for m in plot_methods if m is not None]

labels = ["first_step", "later_steps", "all_seen"]

df_final = df_final[df_final["method"].isin(plot_methods)]

pivot = df_final.pivot_table(
    index="eval_set",
    columns="method",
    values="accuracy",
    aggfunc="mean"
).reindex(labels)

print("Available final methods:", sorted(available_methods))
print("Chosen FT baseline:", ft_name)
print("Pivot columns:", list(pivot.columns))

x = np.arange(len(labels))
width = 0.11
offsets = np.arange(-(len(plot_methods)-1)/2, (len(plot_methods)-1)/2 + 1)

plt.figure(figsize=(14, 6))

for method, offset in zip(plot_methods, offsets):
    if method in pivot.columns:
        plt.bar(x + offset * width, pivot[method], width, label=method)
    else:
        print(f"WARNING: {method} missing in pivot, skipped.")

plt.xticks(x, labels)
plt.ylabel("Accuracy")
plt.title("Final Accuracy Comparison")
plt.legend()
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, "final_accuracy_comparison_all_methods.png")
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved plot to:", plot_path)

In [ ]:
final_first = df_final[df_final["eval_set"] == "first_step"]
final_later = df_final[df_final["eval_set"] == "later_steps"]
final_seen  = df_final[df_final["eval_set"] == "all_seen"]

def make_barplot(df_sub, title, filename):
    methods = ["Merged LoRA", ft_name, "Joint"]
    methods = [m for m in methods if m is not None]

    vals = []
    for m in methods:
        row = df_sub[df_sub["method"] == m]
        vals.append(float(row["accuracy"].iloc[0]) if len(row) > 0 else np.nan)

    plt.figure(figsize=(7, 4))
    plt.bar(methods, vals)
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.tight_layout()

    out_path = os.path.join(PLOTS_DIR, filename)
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved plot to:", out_path)

make_barplot(final_first, "Final Accuracy on B0 Classes", "final_first_step_accuracy_lora_ft_joint.png")
make_barplot(final_later, "Final Accuracy on Later Classes", "final_later_steps_accuracy_lora_ft_joint.png")
make_barplot(final_seen,  "Final Accuracy on All Seen Classes", "final_all_seen_accuracy_lora_ft_joint.png")

In [ ]:
plot_df = results_df_clean.copy()
plot_df_step = plot_df[plot_df["eval_stage"] == "step"].copy()

available_step_methods = set(plot_df_step["method"].astype(str).unique())

if "LoRA + Replay" in available_step_methods:
    lora_method_name = "LoRA + Replay"
elif "LoRA separate" in available_step_methods:
    lora_method_name = "LoRA separate"
else:
    lora_method_name = None

print("Available step methods:", sorted(available_step_methods))
print("Chosen LoRA stepwise method:", lora_method_name)

if lora_method_name is not None:
    lora_current = plot_df_step[
        (plot_df_step["method"] == lora_method_name) &
        (plot_df_step["eval_set"] == "current_step")
    ].copy()

    lora_first = plot_df_step[
        (plot_df_step["method"] == lora_method_name) &
        (plot_df_step["eval_set"] == "first_step")
    ].copy()

    lora_seen = plot_df_step[
        (plot_df_step["method"] == lora_method_name) &
        (plot_df_step["eval_set"] == "all_seen")
    ].copy()

    plt.figure(figsize=(7, 4))
    plt.plot(lora_current["step"], lora_current["accuracy"], marker="o", label="current_step")
    plt.plot(lora_first["step"], lora_first["accuracy"], marker="o", label="first_step")
    plt.plot(lora_seen["step"], lora_seen["accuracy"], marker="o", label="all_seen")
    plt.xticks(lora_current["step"], lora_current["adapter_name"])
    plt.xlabel("Adapter")
    plt.ylabel("Accuracy")
    plt.title(f"{lora_method_name}: Accuracy over L1..L4")
    plt.legend()
    plt.tight_layout()

    out = os.path.join(PLOTS_DIR, "curve_lora_replay_all_metrics.png")
    plt.savefig(out, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", out)
else:
    print("No LoRA stepwise method found for plotting.")

In [ ]:
def make_barplot(df_sub, title, filename):
    methods = [
        "Merged LoRA",
        "Weighted Merged LoRA",
        "Weighted Balanced Merged LoRA",
        "Old-Friendly Merged LoRA",
        "Pairwise Merged LoRA",
        ft_name,
        "Joint"
    ]
    methods = [m for m in methods if m is not None]

    vals = []
    for m in methods:
        row = df_sub[df_sub["method"] == m]
        vals.append(float(row["accuracy"].iloc[0]) if len(row) > 0 else np.nan)

    plt.figure(figsize=(10, 4))
    plt.bar(methods, vals)
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.xticks(rotation=20)
    plt.tight_layout()

    out_path = os.path.join(PLOTS_DIR, filename)
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved plot to:", out_path)

final_first = df_final[df_final["eval_set"] == "first_step"]
final_later = df_final[df_final["eval_set"] == "later_steps"]
final_seen  = df_final[df_final["eval_set"] == "all_seen"]

make_barplot(final_first, "Final Accuracy on B0 Classes", "final_first_step_all_methods.png")
make_barplot(final_later, "Final Accuracy on Later Classes", "final_later_steps_all_methods.png")
make_barplot(final_seen,  "Final Accuracy on All Seen Classes", "final_all_seen_all_methods.png")

In [ ]:
df_merge_only = df_final[df_final["method"].isin([
    "Merged LoRA",
    "Weighted Merged LoRA",
    "Pairwise Merged LoRA"
])].copy()

pivot_merge = df_merge_only.pivot_table(
    index="eval_set",
    columns="method",
    values="accuracy",
    aggfunc="mean"
).reindex(["first_step", "later_steps", "all_seen"])

x = np.arange(3)
width = 0.22

plt.figure(figsize=(9, 5))
plt.bar(x - width, pivot_merge["Merged LoRA"], width, label="Merged LoRA")
plt.bar(x,         pivot_merge["Weighted Merged LoRA"], width, label="Weighted Merged LoRA")
plt.bar(x + width, pivot_merge["Pairwise Merged LoRA"], width, label="Pairwise Merged LoRA")

plt.xticks(x, ["first_step", "later_steps", "all_seen"])
plt.ylabel("Accuracy")
plt.title("Effect of Merge Strategy")
plt.legend()
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, "merge_strategy_comparison.png")
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved plot to:", plot_path)

In [ ]:
merge_summary = df_final[df_final["method"].isin([
    "Merged LoRA",
    "Weighted Merged LoRA",
    "Pairwise Merged LoRA"
])].copy()

print(merge_summary.sort_values(["eval_set", "method"]))